# 01 — Data Cleaning

Pipeline:
1. Load raw `results_linux.csv`
2. Detect & remove outliers (**boxplot / 1.5×IQR fence**, per language × benchmark group, on CPU energy + execution time)
3. Convert units to human-readable values
4. Aggregate to mean per (language, benchmark)
5. Export `results_clean.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

RAW_CSV  = Path('../results/results_linux.csv')
OUT_CSV  = Path('../results/results_clean.csv')
FIGS_DIR = Path('../results/figs')
FIGS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load raw data

In [ ]:
df = pd.read_csv(RAW_CSV)
print(f'Shape: {df.shape}')
df.head(3)

## 2. Outlier detection & removal (Boxplot / 1.5×IQR fence)

Outliers are identified using the standard boxplot rule: any run whose value falls **outside [Q1 − 1.5×IQR, Q3 + 1.5×IQR]** is flagged.  
Detection is applied **per (language × benchmark) group** on the two primary metrics — CPU energy and execution time — to avoid over-flagging from noisy ancillary columns (disk, network, etc.).

In [ ]:
# Numeric metric columns only (exclude identifiers)
ID_COLS     = ['run_id', 'measured_at', 'language', 'benchmark']
METRIC_COLS = [c for c in df.columns if c not in ID_COLS]

# Detect outliers only on the two primary benchmarking signals
KEY_METRICS = [c for c in METRIC_COLS if 'cpu_energy' in c or 'phase_time' in c]
print('Outlier detection on:', KEY_METRICS)

def boxplot_outlier_mask(group: pd.DataFrame, cols: list[str]) -> pd.Series:
    """
    Standard boxplot fence: flag rows outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
    on any of the given columns.
    """
    outlier = pd.Series(False, index=group.index)
    for col in cols:
        q1, q3 = group[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr == 0:          # constant column in this group — skip
            continue
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier |= (group[col] < lo) | (group[col] > hi)
    return outlier

outlier_flags = (
    df.groupby(['language', 'benchmark'], group_keys=False)
      .apply(lambda g: boxplot_outlier_mask(g, KEY_METRICS))
)

n_outliers = outlier_flags.sum()
print(f'Outlier runs flagged : {n_outliers} / {len(df)}  ({n_outliers/len(df)*100:.1f}%)')

df_clean = df[~outlier_flags].copy()
print(f'Rows after removal   : {len(df_clean)}')

In [ ]:
# Visualise outliers — CPU energy boxplots per benchmark, outlier points in red
COL_E      = [c for c in METRIC_COLS if 'cpu_energy' in c][0]
benchmarks = sorted(df['benchmark'].unique())
ncols = 4
nrows = int(np.ceil(len(benchmarks) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()

for i, bm in enumerate(benchmarks):
    ax = axes[i]
    sub   = df[df['benchmark'] == bm].copy()
    sub['_outlier'] = outlier_flags.reindex(sub.index, fill_value=False)
    langs = sorted(sub['language'].unique())

    data_clean   = [sub[(sub['language'] == l) & ~sub['_outlier']][COL_E].values for l in langs]
    data_outlier = [sub[(sub['language'] == l) &  sub['_outlier']][COL_E].values for l in langs]

    bp = ax.boxplot(
        data_clean, labels=langs, patch_artist=True, showfliers=False,
        medianprops=dict(color='black', linewidth=1.5),
        boxprops=dict(facecolor='#aec7e8', alpha=0.8),
        whiskerprops=dict(color='#555555'),
        capprops=dict(color='#555555'),
    )

    # Overlay clean runs as small blue dots, outliers as red crosses
    for j, lang in enumerate(langs):
        x = j + 1
        clean_vals   = data_clean[j]
        outlier_vals = data_outlier[j]
        ax.scatter([x] * len(clean_vals),   clean_vals,   color='#1f77b4', s=8,  alpha=0.5, zorder=3)
        ax.scatter([x] * len(outlier_vals), outlier_vals, color='red',     s=20, alpha=0.9, marker='x', zorder=4, linewidths=1.2)

    ax.set_title(bm, fontsize=9)
    ax.set_ylabel('CPU Energy (µJ)', fontsize=7)
    ax.tick_params(axis='x', rotation=45, labelsize=6)
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

# Legend
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4', markersize=6, label='Clean run'),
    Line2D([0], [0], marker='x', color='red', markersize=8, markeredgewidth=1.5, label='Outlier (boxplot fence)', linestyle='None'),
]
fig.legend(handles=legend_handles, loc='upper right', fontsize=9, frameon=True)

fig.suptitle('CPU Energy Distribution per Benchmark — Boxplot Outlier Detection (1.5×IQR)', y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'boxplot_outliers_cpu_energy.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Per-group count after cleaning — flag any group with fewer than 5 clean runs
counts = df_clean.groupby(['language', 'benchmark']).size().rename('n_runs')
low = counts[counts < 5]
if len(low):
    print('⚠ Groups with < 5 clean runs:')
    print(low.to_string())
else:
    print('All groups have ≥ 5 clean runs.')
counts.unstack('benchmark').fillna(0).astype(int)

In [ ]:
# Visual: distribution of clean run counts per group
fig, ax = plt.subplots(figsize=(5, 3))
counts.value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Runs kept per group')
ax.set_ylabel('Number of groups')
ax.set_title('Clean run counts per (language × benchmark)')
plt.tight_layout()
plt.savefig(FIGS_DIR / 'outlier_counts.pdf', bbox_inches='tight')
plt.show()

## 3. Unit conversions

In [ ]:
# Conversion rules keyed by column-name suffix.
# Each entry: (divisor, new_unit_label)
CONVERSIONS = {
    '-ug' : (1e6,    'g'),        # micro-grams  → grams
    '-uj' : (1e6,    'J'),        # micro-Joules → Joules
    '-mw' : (1e3,    'W'),        # milli-Watts  → Watts
    '-us' : (1e6,    's'),        # micro-seconds → seconds
    '-bytes': (1e6,  'MB'),       # bytes → MB  (disk/net totals)
}

rename_map = {}
for col in METRIC_COLS:
    for suffix, (divisor, unit) in CONVERSIONS.items():
        if col.endswith(suffix):
            base = col[: -len(suffix)]
            new_col = f'{base}-{unit.lower()}'
            df_clean[new_col] = df_clean[col] / divisor
            rename_map[col] = new_col
            break

# Drop the original raw-unit columns
df_clean = df_clean.drop(columns=list(rename_map.keys()))

METRIC_COLS_CONV = [c for c in df_clean.columns if c not in ID_COLS]
print('Converted columns:')
for old, new in rename_map.items():
    print(f'  {old}  →  {new}')

## 4. Aggregate to mean per (language, benchmark)

In [ ]:
df_mean = (
    df_clean
      .groupby(['language', 'benchmark'])[METRIC_COLS_CONV]
      .mean()
      .reset_index()
)
print(f'Aggregated shape: {df_mean.shape}')
df_mean.head()

## 5. Export

In [ ]:
df_mean.to_csv(OUT_CSV, index=False)
print(f'Saved → {OUT_CSV}')

# Also save the per-run cleaned frame (before aggregation) for reference
df_clean.to_csv(OUT_CSV.with_name('results_clean_runs.csv'), index=False)
print(f'Saved → {OUT_CSV.with_name("results_clean_runs.csv")}')